In [1]:
#undef __noinline__

In [2]:
#define N 10000000

__global__ void vector_add( int *a, int *b, int *c) {
     int tid = blockIdx.x * blockDim.x + threadIdx.x;
     if (tid < N)
         c[tid] = a[tid] + b[tid];
}

__global__ void vector_add_block( int *a, int *b, int *c ) {
     int tid = blockIdx.x;
     if (tid < N)
     c[tid] = a[tid] + b[tid];
}

In [3]:
int v_a[N], v_b[N], v_c[N];
int *dev_a, *dev_b, *dev_c;

cudaEvent_t start, stop;

cudaEventCreate(&start);
cudaEventCreate(&stop);

// allocate the memory on the GPU
cudaMalloc( (void**)&dev_a, N * sizeof(int) );
cudaMalloc( (void**)&dev_b, N * sizeof(int) );
cudaMalloc( (void**)&dev_c, N * sizeof(int) );

// fill the arrays 'a' and 'b' on the CPU
for (int i=0; i<N; i++) {
    v_a[i] = i;
    v_b[i] = i * i;
}

// copy the arrays 'a' and 'b' to the GPU
cudaMemcpy( dev_a, v_a, N * sizeof(int), cudaMemcpyHostToDevice );
cudaMemcpy( dev_b, v_b, N * sizeof(int), cudaMemcpyHostToDevice );

In [4]:
float time_block_method = 0;
float time_thread_method = 0;


cudaEventRecord(start);

vector_add_block<<<N, 1>>>(dev_a, dev_b, dev_c); 

cudaEventRecord(stop);
cudaEventSynchronize(stop);
cudaEventElapsedTime(&time_block_method, start, stop);

In [5]:
int threadsPerBlock = 256;
int blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock;

cudaEventRecord(start);

vector_add<<<blocksPerGrid, threadsPerBlock>>>(dev_a, dev_b, dev_c); 

cudaEventRecord(stop);
cudaEventSynchronize(stop);
cudaEventElapsedTime(&time_thread_method, start, stop);

In [6]:
cudaEventDestroy(start);
cudaEventDestroy(stop);

cudaFree( dev_a );
cudaFree( dev_b );
cudaFree( dev_c );

In [7]:
#include <iostream>
#include <vector>
#include <chrono>


void add_CPU(const int* a, const int* b, int* c, int n) {
    for (int i = 0; i < n; i++) {
        c[i] = a[i] + b[i];
    }
}

    // Using std::vector for easy memory management on the CPU heap
    std::vector<int> h_a(N), h_b(N), h_c(N);

    // Initialize arrays
    for (int i = 0; i < N; i++) {
        h_a[i] = i;
        h_b[i] = i * 2;
    }

    // Start Timer
    auto cstart = std::chrono::high_resolution_clock::now();

    add_CPU(h_a.data(), h_b.data(), h_c.data(), N);

    // Stop Timer
    auto cend = std::chrono::high_resolution_clock::now();

    // Calculate duration in milliseconds
    std::chrono::duration<double, std::milli> duration = cend - cstart;

In [8]:
printf("Time (CPU sort): %f ms\n", duration.count());

printf("Time (1 thread/block): %f ms\n", time_block_method);

printf("Time (256 threads/block): %f ms\n", time_thread_method);

Time (CPU sort): 19.924253 ms
Time (1 thread/block): 31.067553 ms
Time (256 threads/block): 1.769984 ms
